# Day 2 - Từ annotation YOLO đến detector YOLO26

Notebook này chạy **sau** khi nhãn 50 ảnh train đã qua cổng gold-set. Tập `eval` 10 ảnh không được dùng để train.

Quy trình chuẩn: bạn **fork repo đề bài**, gán nhãn, **commit và push** `dataset/labels/train/` lên repo của mình, rồi chỉ cần dán URL repo vào `REPO_URL` ở cell đầu tiên — Colab sẽ tự clone repo, train và đánh giá, không cần upload file.

Bài cá nhân. Hạn nộp: 11:59 trưa ngày hôm sau buổi lab (UTC+7); ca 09/11/2026 nộp trước 11:59 ngày 10/11/2026.

## Kiểm tra trước khi chạy

- Đã **fork repo đề bài**, gán nhãn và **commit + push** `dataset/labels/train/` lên repo của bạn.
- Ở cell tiếp theo, dán **URL repo của bạn** vào `REPO_URL` — notebook tự clone, không cần upload file.
- Không có `gold_train_labels/` trong repo.
- Chạy từ trên xuống dưới; giữ nguyên `data.yaml` và split eval.
- mAP đo model; F1/IoU ở script trước đó đo chất lượng nhãn con người.
- Eval raw ID khác thứ tự schema student; notebook chuẩn hóa ID trên bản sao runtime, giữ nguyên ảnh, box và split nguồn.
- Phiên chạy này train nhãn student sau rework. Kết quả gold_reference được lưu riêng trong day2_gold_results.ipynb.


In [ ]:
%pip -q install ultralytics==8.4.208 pyyaml

from pathlib import Path
import importlib.util
import json
import os
import subprocess
import torch
from ultralytics import YOLO

REPO_URL = 'https://github.com/trxuanbac/K5-L2A-DAY02-TranXuanBac-2A202603132-ObjectDetection'
LAB_ROOT = (Path('/content/Day2-Lab') if Path('/content').is_dir() else Path('Day2-Lab')).resolve()
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
if LAB_ROOT.exists():
    # Giữ các thay đổi local: nếu repo có xung đột, git sẽ dừng để kiểm tra.
    subprocess.run(['git', '-C', str(LAB_ROOT), 'pull', '--ff-only'], check=True, env=env)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(LAB_ROOT)], check=True, env=env)
print('Repo commit:', subprocess.check_output(['git', '-C', str(LAB_ROOT), 'rev-parse', 'HEAD'], text=True).strip())

TRAIN_IMAGES = LAB_ROOT / 'dataset/images/train'
TRAIN_LABELS = LAB_ROOT / 'dataset/labels/train'
EVAL_IMAGES = LAB_ROOT / 'dataset/images/eval'
EVAL_LABELS = LAB_ROOT / 'dataset/labels/eval'
RUNS = LAB_ROOT / 'outputs/yolo26_runs'
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


In [ ]:
def image_stems(directory):
    return {p.stem for ext in ('*.jpg', '*.jpeg', '*.png') for p in directory.glob(ext)}

train_images = image_stems(TRAIN_IMAGES)
train_labels = {p.stem for p in TRAIN_LABELS.glob('*.txt')}
eval_images = image_stems(EVAL_IMAGES)
eval_labels = {p.stem for p in EVAL_LABELS.glob('*.txt')}
assert len(train_images) == 50, f' cần 50 train images, thấy {len(train_images)}'
assert train_images == train_labels, f'train images/labels lệch: {sorted(train_images ^ train_labels)[:5]}'
assert len(eval_images) == 10 and eval_images == eval_labels, 'eval phải đủ 10 ảnh và nhãn'
print(f'OK: {len(train_images)} train + {len(eval_images)} eval; eval không đi vào train.')
# CP5 phải đạt trên đủ 50 ảnh trước khi train; không tạo JSON minh họa.
QC_PATH = LAB_ROOT / 'outputs/annotation_evaluation.json'
assert QC_PATH.is_file(), 'Chưa có kết quả gold QC. Khóa nhãn, nhận gold đúng mốc và chạy evaluator trước.'
annotation_qc = json.loads(QC_PATH.read_text(encoding='utf-8'))
assert annotation_qc.get('passed') is True, 'Gold QC chưa đạt: cần rà và rework nhãn trước khi train.'
assert annotation_qc.get('f1', 0) >= 0.80 and annotation_qc.get('mean_iou_of_matches', 0) >= 0.70, 'Chưa đạt cả hai ngưỡng F1 và mean IoU.'
assert annotation_qc.get('iou_match_threshold') == 0.50, 'Matching phải dùng IoU 0.50 theo bài.'
assert len(annotation_qc.get('per_image', [])) == 50, 'Gold QC phải bao phủ đủ 50 ảnh train.'
assert {x['image'] for x in annotation_qc['per_image']} == train_images, 'Gold QC không khớp tập train.'
print('Gold QC đạt trên đủ 50 ảnh; được phép train YOLO26n.')

# Khóa đúng nhãn đã qua CP5 bằng SHA-256; giữ split và các file eval nguồn.
# Eval phát hành dùng raw ID 0=car,1=van,2=truck,3=bus.
# Script chỉ đổi ID trên bản sao runtime sang schema student 0=car,1=truck,2=bus,3=van.
prep_path = LAB_ROOT / 'tools/prepare_student_dataset.py'
spec = importlib.util.spec_from_file_location('prepare_student_dataset', prep_path)
prep = importlib.util.module_from_spec(spec)
spec.loader.exec_module(prep)
DATA_YAML = prep.prepare(LAB_ROOT)
print('Runtime dataset:', DATA_YAML)
print('Giữ nguyên eval nguồn; train lấy đúng 50 file nhãn đã đạt CP5.')


## Huấn luyện

Bài thực hành dùng checkpoint `yolo26n.pt`. Với 50 ảnh, dùng model nano và ít epoch để minh họa vòng lặp dữ liệu; thời gian/metric không dùng làm benchmark sản phẩm.

In [ ]:
model = YOLO('yolo26n.pt')
train_result = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=-1,
    device=DEVICE,
    project=str(RUNS),
    name='vehicle_annotation',
    exist_ok=True,
    seed=20260909,
    plots=True,
)
BEST = Path(model.trainer.best)
print('best checkpoint:', BEST)

## Đánh giá đúng tập eval

Runtime YAML trỏ `val` tới bản sao đúng 10 ảnh eval và nhãn đã chuẩn hóa ID về schema student. Box và split eval giữ nguyên. Ghi lại mAP50, mAP50–95, precision và recall.

Eval cũng dùng làm validation chọn `best.pt`, nên không phải test set độc lập.


In [ ]:
best_model = YOLO(str(BEST))
metrics = best_model.val(data=str(DATA_YAML), split='val', device=DEVICE, plots=True, project=str(RUNS), name='vehicle_eval', exist_ok=True)
summary = {
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'eval_images': len(eval_images),
    'checkpoint': str(BEST),
}
metrics_path = LAB_ROOT / 'outputs/model_eval_metrics.json'
metrics_path.parent.mkdir(parents=True, exist_ok=True)
metrics_path.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))

In [ ]:
# Kiểm tra trực quan một vài ảnh eval; đây là bước bắt buộc trong báo cáo.
prediction_dir = LAB_ROOT / 'outputs/eval_predictions'
results = best_model.predict(source=str(EVAL_IMAGES), conf=0.25, device=DEVICE, save=True, project=str(prediction_dir), name='visuals', exist_ok=True)
PRED_SAVE_DIR = Path(results[0].save_dir)
print('predictions saved to:', PRED_SAVE_DIR)
# Chỉ đóng gói artifact cần nộp; tránh ZIP tự chứa chính nó khi chạy lại.
from zipfile import ZipFile, ZIP_DEFLATED
archive = LAB_ROOT / 'outputs/day2_colab_outputs.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as package:
    for directory in (prediction_dir, RUNS):
        for artifact in sorted(directory.rglob('*')):
            if artifact.is_file():
                package.write(artifact, 'day2_colab_outputs/' + artifact.relative_to(LAB_ROOT / 'outputs').as_posix())
    package.write(metrics_path, 'day2_colab_outputs/model_eval_metrics.json')
print('Đã tạo:', archive)


## Xem trực tiếp kết quả dự đoán

Hiển thị 10 ảnh eval kèm box dự đoán ngay trong notebook, không cần tải file `.zip` về mới xem được.

In [ ]:
import math
import matplotlib.pyplot as plt
from PIL import Image

pred_images = sorted(PRED_SAVE_DIR.glob('*.jpg'))
cols = 5
rows = math.ceil(len(pred_images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
for ax, img_path in zip(axes.flat, pred_images):
    ax.imshow(Image.open(img_path))
    ax.set_title(img_path.name, fontsize=9)
    ax.axis('off')
for ax in axes.flat[len(pred_images):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

## Câu hỏi phân tích để nộp

1. Lớp nào có recall thấp nhất? Chỉ ra một ảnh eval minh họa.
2. Nếu mAP50 cao nhưng mAP50-95 thấp, box đang gặp vấn đề gì?
3. Có lỗi `truck`/`bus`/`van`/`car` nào liên quan tới guideline không?
4. Vì sao không dùng 10 ảnh eval để train dù chúng có nhãn?